# Lubb | تدريب نموذج بوصلة القرار الذكي

دفتر تدريب K-Means مع نتائج التشغيل الفعلية المحفوظة. المصدر: `Lubb_Clusters_Normalized_Retrained_1282.csv` (1,282 سجلًا). يستخدم العمر والحالة الاجتماعية والفئة التعليمية. الأرقام في الداشبورد: Silhouette=0.314، ARI=0.917، أحجام المجموعات 368، 265، 240، 236، 173.

**تنبيه:** نتائج التجميع استكشافية ولا تصلح وحدها لاتخاذ قرارات رعاية. احتفظ بملف البيانات بجوار الدفتر عند إعادة التشغيل.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
import joblib

INPUT = Path('Lubb_Clusters_Normalized_Retrained_1282.csv')
df = pd.read_csv(INPUT).drop(columns=['المجموعة'], errors='ignore')
assert len(df) == 1282
print('عدد السجلات:', len(df))

عدد السجلات: 1282


## 1. توحيد القيم وترميز الخصائص

In [2]:
df['الحالة الاجتماعية'] = df['الحالة الاجتماعية'].replace({'مطلق/مطلقة':'مطلق/ة','متزوج/متزوجة':'متزوج/ة'})
scaler = StandardScaler()
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X = np.column_stack([
    scaler.fit_transform(df[['العمر']]),
    encoder.fit_transform(df[['الحالة الاجتماعية','الفئة التعليمية']].fillna('غير محدد'))
])
print('شكل الخصائص:', X.shape)

شكل الخصائص: (1282, 17)


## 2. تدريب K-Means وترقيم المجموعات من الأكبر إلى الأصغر

In [3]:
model = KMeans(n_clusters=5, random_state=42, n_init=20)
labels = model.fit_predict(X)
sizes = pd.Series(labels).value_counts().sort_values(ascending=False)
label_map = {int(old): i+1 for i,old in enumerate(sizes.index)}
df['المجموعة'] = pd.Series(labels).map(label_map)
print(df['المجموعة'].value_counts().sort_index().to_string())

المجموعة
1    368
2    265
3    240
4    236
5    173


## 3. التقييم

ARI المعروض في الداشبورد هو **متوسط ARI مقابل التشغيل الأساسي** عند استخدام البذور `[42,7,19,31,53]` و`n_init=20`، وليس متوسط جميع أزواج التشغيلات.

In [4]:
sil = silhouette_score(X, labels)
seeds = [42, 7, 19, 31, 53]
runs = [KMeans(n_clusters=5,random_state=s,n_init=20).fit_predict(X) for s in seeds]
ari = np.mean([adjusted_rand_score(runs[0],r) for r in runs[1:]])
ari_pairwise = np.mean([adjusted_rand_score(a,b) for a,b in combinations(runs,2)])
print(f'Silhouette: {sil:.3f}')
print(f'ARI مقابل التشغيل الأساسي: {ari:.3f}')
print(f'ARI لجميع الأزواج (مقياس مختلف): {ari_pairwise:.3f}')

Silhouette: 0.314
ARI مقابل التشغيل الأساسي: 0.917
ARI لجميع الأزواج (مقياس مختلف): 0.927


## 4. متوسط العمر وحفظ النتائج

In [5]:
print(df.groupby('المجموعة').agg(عدد_السجلات=('العمر','size'),متوسط_العمر=('العمر','mean')).round(1).to_string())
df.to_csv('Lubb_Clusters_Reproduced_1282.csv',index=False,encoding='utf-8-sig')
joblib.dump({'model':model,'scaler':scaler,'encoder':encoder,'label_map':label_map},'Lubb_KMeans_Model.joblib')
print('تم حفظ ملف النتائج والنموذج')

          عدد_السجلات  متوسط_العمر
المجموعة                          
1                 368         65.3
2                 265         67.2
3                 240         86.4
4                 236         87.3
5                 173         65.5
تم حفظ ملف النتائج والنموذج
